In [1]:
import keras
print(keras.__version__)
print(keras.__file__)

3.5.0
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/__init__.py


In [2]:
import tensorflow as tf
import numpy as np
import cv2
from PIL import Image
import os

def load_model(model_dir):
    arch_path = os.path.join(model_dir, "arch.json")
    weights_path = os.path.join(model_dir, "weights.h5")

    # Load architecture
    with open(arch_path, "r") as f:
        model = tf.keras.models.model_from_json(f.read())

    # Load weights
    model.load_weights(weights_path)
    return model

def resize_image_to_fit(image, target_size):
    h, w = image.shape[:2]
    pad_h = max(target_size - h, 0)
    pad_w = max(target_size - w, 0)
    image = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=255)
    return image[:target_size, :target_size]

def preprocess_image(image_path, target_size):
    image = Image.open(image_path).convert("RGB")
    image = np.array(image)
    image = resize_image_to_fit(image, target_size)
    image = image.astype(np.uint8)
    return np.expand_dims(image, axis=0)

def postprocess_prediction(prediction):
    # Assuming 3 output classes: background, staffline, symbols
    class_map = np.argmax(prediction[0], axis=-1)
    return class_map

# ---- MAIN ----
model_dir = "/Users/philip/Projects/E:ACC/MuseScore-Plus/u_net"
image_path = "/Users/philip/Projects/E:ACC/MuseScore-Plus/test_image.png"  # change this to your input
window_size = 256  # likely input size used in U-Net training

# 1. Load model
model = load_model(model_dir)

# 2. Preprocess image
input_image = preprocess_image(image_path, window_size)

# 3. Predict
prediction = model.predict(input_image)

# 4. Postprocess output
class_map = postprocess_prediction(prediction)

# 5. Visualize result
Image.fromarray((class_map * 127).astype(np.uint8)).show()


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/Users/philip/Projects/E:ACC/MuseScore-Plus/u_net/weights.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)